# Detyre Shtepie — Analiza e Kompleksitetit Kohor

---

## Konteksti

Ne laborator kemi krahasuar tre algoritmet e renditjes duke matur kohen ne sekonda.
Siç kemi diskutuar, koha ne sekonda nuk eshte metrika e sakte per kompleksitetin kohor
sepse varet nga hardware-i, gjuha programuese dhe faktore te tjere te jashtem.

Metrika e sakte eshte numri i krahasimeve:  i pavarur nga çdo faktor i jashtem.

---

## Detyra

Kodi i meposhtem eshte pothuajse identik me ate te laboratorit, me dy shtesa:

- Jane shtuar funksionet `insertion_sort_ops`, `selection_sort_ops`, `merge_sort_ops`
  dhe funksioni `mat_operacionet`, te gatshem, nuk i ndryshoni.

E vetmja gje qe duhet te beni: Ne kodin e llogaritjes, zevendesoni
`mat_kohen` me `mat_operacionet` dhe funksionet e vjetra me ato `_ops`.
Shikoni komentet `# NDRYSHO` — jane saktesisht 3 rreshta per tu ndryshuar.

---

In [ ]:
import time
import random
import plotly.graph_objects as go


# Algoritmet origjinale (qe perdorem ne laboratore)

def insertion_sort(lista):
    n = len(lista)
    for i in range(1, n):
        vlera = lista[i]
        j = i - 1
        while j >= 0 and lista[j] > vlera:
            lista[j + 1] = lista[j]
            j -= 1
        lista[j + 1] = vlera


def selection_sort(lista):
    n = len(lista)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if lista[j] < lista[min_idx]:
                min_idx = j
        lista[i], lista[min_idx] = lista[min_idx], lista[i]


def merge_sort(lista):
    if len(lista) <= 1:
        return lista
    mid   = len(lista) // 2
    left  = merge_sort(lista[:mid])
    right = merge_sort(lista[mid:])
    res, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            res.append(left[i])
            i += 1
        else:
            res.append(right[j])
            j += 1
    res.extend(left[i:])
    res.extend(right[j:])
    return res


# Algoritmet qe numerojne krahasimet (mos i ndryshoni, ato qe do perdorni)

def insertion_sort_ops(lista):
    n   = len(lista)
    ops = 0
    for i in range(1, n):
        vlera = lista[i]
        j     = i - 1
        while j >= 0 and lista[j] > vlera:
            lista[j + 1] = lista[j]
            j   -= 1
            ops += 1
        if j >= 0:
            ops += 1
        lista[j + 1] = vlera
    return ops


def selection_sort_ops(lista):
    n   = len(lista)
    ops = 0
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            ops += 1
            if lista[j] < lista[min_idx]:
                min_idx = j
        lista[i], lista[min_idx] = lista[min_idx], lista[i]
    return ops


def merge_sort_ops(lista):
    ops = [0]
    def _sort(arr):
        if len(arr) <= 1:
            return arr
        mid   = len(arr) // 2
        left  = _sort(arr[:mid])
        right = _sort(arr[mid:])
        res, i, j = [], 0, 0
        while i < len(left) and j < len(right):
            ops[0] += 1
            if left[i] <= right[j]:
                res.append(left[i])
                i += 1
            else:
                res.append(right[j])
                j += 1
        res.extend(left[i:])
        res.extend(right[j:])
        return res
    _sort(lista)
    return ops[0]


# Funksioni i matjes se kohes (i vjeter)

def mat_kohen(funksioni, lista):
    kopje = lista.copy()
    t     = time.perf_counter()
    funksioni(kopje)
    return time.perf_counter() - t


# Funksioni i matjes se krahasimeve (ai qe do perdorni ju)

def mat_operacionet(funksioni, lista):
    kopje = lista.copy()
    return funksioni(kopje)


# Llogaritja e rezultateve
#  NDRYSHO — zevendeso 3 rreshtat:
#    mat_kohen(insertion_sort, ...)  zevendesoje me    mat_operacionet(insertion_sort_ops, ...)
#    mat_kohen(selection_sort, ...)  zevendesoje me    mat_operacionet(selection_sort_ops, ...)
#    mat_kohen(merge_sort,     ...)  zevendesoje me    mat_operacionet(merge_sort_ops,     ...)
#
# ─────────────────────────────────────────────────────────────────────────────

madhesite = [10, 50, 100, 500, 1000, 2000, 5000]
raste     = ["Random", "E renditur", "Reverse"]
colors    = {
    "Insertion Sort": "#FF8C46",
    "Selection Sort": "#50B9FF",
    "Merge Sort":     "#50E178",
}

rezultatet = {alg: {r: [] for r in raste} for alg in colors}

for n in madhesite:
    listat = {
        "Random":      random.sample(range(n * 10), n),
        "E renditur":  list(range(n)),
        "Reverse":     list(range(n, 0, -1)),
    }
    for rast, lst in listat.items():
        rezultatet["Insertion Sort"][rast].append(mat_kohen(insertion_sort, lst))  # NDRYSHO
        rezultatet["Selection Sort"][rast].append(mat_kohen(selection_sort, lst))  # NDRYSHO
        rezultatet["Merge Sort"][rast].append(mat_kohen(merge_sort, lst))          # NDRYSHO


fig = go.Figure()

for alg, color in colors.items():
    for rast in raste:
        vlerat = rezultatet[alg][rast]
        hover  = [
            f"{alg}<br>Rasti: {rast}<br>n={n}<br>{vlerat[i]}"
            for i, n in enumerate(madhesite)
        ]
        dashja = "solid" if rast == "Random" else ("dot" if rast == "E renditur" else "dash")
        fig.add_trace(go.Scatter(
            x     = madhesite,
            y     = vlerat,
            name  = f"{alg} — {rast}",
            line  = dict(color=color, width=2, dash=dashja),
            mode  = "lines+markers",
            marker= dict(size=7),
            text  = hover,
            hovertemplate="%{text}<extra></extra>"
        ))

fig.update_layout(
    title       = "Insertion vs Selection vs Merge Sort — Numri i Krahasimeve",
    xaxis       = dict(title="n (madhësia e listës)", gridcolor="#eeeeee"),
    yaxis       = dict(title="Numri i krahasimeve",   gridcolor="#eeeeee"),
    plot_bgcolor= "white",
    hoverlabel  = dict(bgcolor="#1e1e2e", font_color="white", font_size=11),
    legend      = dict(groupclick="toggleitem")
)
fig.show()

---

## Pyetjet e Analizes

Bazohuni ne grafikun e gjeneruar dhe ne njohurite e fituara gjate laboratoreve.

---

**Pyetja 1.**  
Shikoni tre vijat e **Selection Sort** ne grafik (Random, E renditur, Reverse - ne grafik shikoni me kujdes qe vijat mbishkruajne njera tjetren).  
Çfare vezhgoni? Shpjegoni pse ndodh kjo.

*Pergjigja juaj:*
> *(shkruani ketu)*

---

**Pyetja 2.**  
Per **Insertion Sort**, cili rast ben me pak krahasime — *E renditur* apo *Reverse* (shikoni me kujdes qe reverse ndodhet aty ku ndodhet Selection)?  
Pse ndodh kjo diferenca?

*Pergjigja juaj:*
> *(shkruani ketu)*

---

**Pyetja 3.**  
Per n = 1000, krahasoni numrin e krahasimeve te **Merge Sort** me ate te **Selection Sort** (rasti Random).  
Sa here me pak krahasime ben Merge Sort? Çfare tregon kjo?

*Pergjigja juaj:*
> *(shkruani ketu)*

---

**Pyetja 4.**  
Dy algoritme jane te dy **O(n²)**. A do te thote kjo se bejne saktesisht te njejtin numer krahasimesh?  
Argumentoni me nje shembull konkret nga grafiku.

*Pergjigja juaj:*
> *(shkruani ketu)*

---

**Pyetja 5.**  
Pse kemi zevendesuar matjen e **sekondave** me matjen e **krahasimeve**?  
Jepni dy arsye.

*Pergjigja juaj:*
> *(shkruani ketu)*